In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

PROJECT_ROOT = Path("/mnt/d/IDC409_Project15")
DB_PATH      = PROJECT_ROOT / "data" / "processed" / "house_prices.db"

engine = create_engine(f"sqlite:///{DB_PATH}")
df = pd.read_sql("SELECT * FROM house_prices_clean", con=engine)

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {df.columns.tolist()}")

Loaded: 250,000 rows × 24 columns

Columns: ['State', 'City', 'Locality', 'Property_Type', 'BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt', 'Furnished_Status', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Facing', 'Owner_Type', 'Availability_Status', 'Amenities_Count', 'Has_Gym', 'Has_Pool', 'Has_Garden', 'Has_Playground', 'Has_Clubhouse']


In [2]:
cols_to_drop = [
    'Locality',        # too many unique values for encoding
    'Price_per_SqFt',  # data leakage (derived from the target)
]

df = df.drop(columns=cols_to_drop)
print(f"After dropping: {df.shape[1]} columns")
print(f"Dropped: {cols_to_drop}")

After dropping: 22 columns
Dropped: ['Locality', 'Price_per_SqFt']


In [3]:
# Always separate X (features) and y (target) early
# This makes everything cleaner going forward

TARGET = 'Price_in_Lakhs'

y = df[TARGET].copy()
X = df.drop(columns=[TARGET]).copy()

print(f"Target  (y): {y.shape[0]:,} values")
print(f"Features (X): {X.shape[1]} columns")
print(f"\nTarget stats:")
print(y.describe())

Target  (y): 250,000 values
Features (X): 21 columns

Target stats:
count    250000.000000
mean        254.586854
std         141.349921
min          10.000000
25%         132.550000
50%         253.870000
75%         376.880000
max         500.000000
Name: Price_in_Lakhs, dtype: float64


In [4]:
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()

print(f"Numeric columns ({len(num_cols)}) — ready to use:")
for c in num_cols:
    print(f"  {c}")

print(f"\nCategorical columns ({len(cat_cols)}) — need encoding:")
for c in cat_cols:
    print(f"  {c:35s} {X[c].nunique()} unique values: {X[c].unique()[:5].tolist()}")

Numeric columns (11) — ready to use:
  BHK
  Size_in_SqFt
  Age_of_Property
  Nearby_Schools
  Nearby_Hospitals
  Amenities_Count
  Has_Gym
  Has_Pool
  Has_Garden
  Has_Playground
  Has_Clubhouse

Categorical columns (10) — need encoding:
  State                               20 unique values: ['Tamil Nadu', 'Maharashtra', 'Punjab', 'Rajasthan', 'West Bengal']
  City                                42 unique values: ['Chennai', 'Pune', 'Ludhiana', 'Jodhpur', 'Jaipur']


/tmp/ipykernel_3744/4072160397.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include='object').columns.tolist()


  Property_Type                       3 unique values: ['Apartment', 'Independent House', 'Villa']
  Furnished_Status                    3 unique values: ['Furnished', 'Unfurnished', 'Semi-furnished']
  Public_Transport_Accessibility      3 unique values: ['High', 'Low', 'Medium']
  Parking_Space                       2 unique values: ['No', 'Yes']
  Security                            2 unique values: ['No', 'Yes']
  Facing                              4 unique values: ['West', 'North', 'South', 'East']
  Owner_Type                          3 unique values: ['Owner', 'Builder', 'Broker']
  Availability_Status                 2 unique values: ['Ready_to_Move', 'Under_Construction']


In [5]:
# One-hot encoding: converts each category into a separate 0/1 column
# Example: Furnished_Status with values [Furnished, Unfurnished, Semi-Furnished]
# becomes three columns: Furnished_Status_Furnished, 
#                        Furnished_Status_Unfurnished,
#                        Furnished_Status_Semi-Furnished
#
# drop_first=True: drops one column per feature to avoid redundancy
# (if Furnished=0 AND Unfurnished=0, we already know it's Semi-Furnished)
# This is called the "dummy variable trap"

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(f"Before encoding : {X.shape[1]} columns")
print(f"After encoding  : {X_encoded.shape[1]} columns")
print(f"\nNew columns added by encoding:")
new_cols = [c for c in X_encoded.columns if c not in num_cols]
for c in new_cols:
    print(f"  {c}")

Before encoding : 21 columns
After encoding  : 85 columns

New columns added by encoding:
  State_Assam
  State_Bihar
  State_Chhattisgarh
  State_Delhi
  State_Gujarat
  State_Haryana
  State_Jharkhand
  State_Karnataka
  State_Kerala
  State_Madhya Pradesh
  State_Maharashtra
  State_Odisha
  State_Punjab
  State_Rajasthan
  State_Tamil Nadu
  State_Telangana
  State_Uttar Pradesh
  State_Uttarakhand
  State_West Bengal
  City_Amritsar
  City_Bangalore
  City_Bhopal
  City_Bhubaneswar
  City_Bilaspur
  City_Chennai
  City_Coimbatore
  City_Cuttack
  City_Dehradun
  City_Durgapur
  City_Dwarka
  City_Faridabad
  City_Gaya
  City_Gurgaon
  City_Guwahati
  City_Haridwar
  City_Hyderabad
  City_Indore
  City_Jaipur
  City_Jamshedpur
  City_Jodhpur
  City_Kochi
  City_Kolkata
  City_Lucknow
  City_Ludhiana
  City_Mangalore
  City_Mumbai
  City_Mysore
  City_Nagpur
  City_New Delhi
  City_Noida
  City_Patna
  City_Pune
  City_Raipur
  City_Ranchi
  City_Silchar
  City_Surat
  City_Trivandr

In [6]:
non_numeric = X_encoded.select_dtypes(include='object').columns.tolist()

if len(non_numeric) == 0:
    print("All columns are numeric — ready for ML models.")
else:
    print(f"WARNING: These columns are still text and need fixing: {non_numeric}")

print(f"\nFinal feature matrix shape: {X_encoded.shape}")
print(f"dtype counts:")
print(X_encoded.dtypes.value_counts())

All columns are numeric — ready for ML models.

Final feature matrix shape: (250000, 85)
dtype counts:
bool     74
int64    11
Name: count, dtype: int64


In [7]:
# pd.get_dummies sometimes creates bool columns — models prefer int (0/1)
bool_cols = X_encoded.select_dtypes(include='bool').columns.tolist()

if bool_cols:
    X_encoded[bool_cols] = X_encoded[bool_cols].astype(int)
    print(f"Converted {len(bool_cols)} bool columns to int: {bool_cols}")
else:
    print("No bool columns found — nothing to convert.")

# Final dtype check
print(f"\nAll dtypes now:")
print(X_encoded.dtypes.value_counts())

Converted 74 bool columns to int: ['State_Assam', 'State_Bihar', 'State_Chhattisgarh', 'State_Delhi', 'State_Gujarat', 'State_Haryana', 'State_Jharkhand', 'State_Karnataka', 'State_Kerala', 'State_Madhya Pradesh', 'State_Maharashtra', 'State_Odisha', 'State_Punjab', 'State_Rajasthan', 'State_Tamil Nadu', 'State_Telangana', 'State_Uttar Pradesh', 'State_Uttarakhand', 'State_West Bengal', 'City_Amritsar', 'City_Bangalore', 'City_Bhopal', 'City_Bhubaneswar', 'City_Bilaspur', 'City_Chennai', 'City_Coimbatore', 'City_Cuttack', 'City_Dehradun', 'City_Durgapur', 'City_Dwarka', 'City_Faridabad', 'City_Gaya', 'City_Gurgaon', 'City_Guwahati', 'City_Haridwar', 'City_Hyderabad', 'City_Indore', 'City_Jaipur', 'City_Jamshedpur', 'City_Jodhpur', 'City_Kochi', 'City_Kolkata', 'City_Lucknow', 'City_Ludhiana', 'City_Mangalore', 'City_Mumbai', 'City_Mysore', 'City_Nagpur', 'City_New Delhi', 'City_Noida', 'City_Patna', 'City_Pune', 'City_Raipur', 'City_Ranchi', 'City_Silchar', 'City_Surat', 'City_Trivandr

In [8]:
from sklearn.model_selection import train_test_split

# 80% training, 20% testing
# random_state=42 means the split is reproducible — run it again, get same split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

print(f"Training set   : {X_train.shape[0]:,} rows")
print(f"Test set       : {X_test.shape[0]:,} rows")
print(f"Features       : {X_train.shape[1]} columns")
print(f"\nTraining target mean : ₹{y_train.mean():.2f}L")
print(f"Test target mean     : ₹{y_test.mean():.2f}L")
print("(These should be close — confirms the split is balanced)")

Training set   : 200,000 rows
Test set       : 50,000 rows
Features       : 85 columns

Training target mean : ₹254.55L
Test target mean     : ₹254.71L
(These should be close — confirms the split is balanced)


In [9]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Save the model-ready features and target
X_train.to_csv(PROCESSED_DIR / "X_train.csv", index=False)
X_test.to_csv(PROCESSED_DIR  / "X_test.csv",  index=False)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR  / "y_test.csv",  index=False)

# Also save to SQLite
X_encoded_with_target = X_encoded.copy()
X_encoded_with_target[TARGET] = y.values
X_encoded_with_target.to_sql("house_prices_model_ready",
                               con=engine, if_exists="replace", index=False)

print("Saved:")
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train : {y_train.shape}")
print(f"  y_test  : {y_test.shape}")
print(f"  SQLite table: house_prices_model_ready")

Saved:
  X_train : (200000, 85)
  X_test  : (50000, 85)
  y_train : (200000,)
  y_test  : (50000,)
  SQLite table: house_prices_model_ready


## Phase 4 Summary — Feature Engineering

**Dropped columns:** Locality (high cardinality), Price_per_SqFt (data leakage)

**One-hot encoded:** State, City, Property_Type, Furnished_Status,
Public_Transport_Accessibility, Parking_Space, Security, Facing,
Owner_Type, Availability_Status

**Final feature count:** Check Cell 5 output

**Train/test split:** 80% train / 20% test, random_state=42

**Data leakage prevention:** Price_per_SqFt excluded from all model inputs.
This column cannot be known before knowing the price itself.

**Ready for modelling:** X_train, X_test, y_train, y_test saved to
data/processed/ and SQLite.